# i. Introduction

================================================= <br>
**Data-Driven Term Deposit Subscription Modeling for Marketing Campaigns** (*Model Inference*)

Name : Fernando Namora <br>
Date Creation : February 4, 2026

This notebook is for model inference to predict new clients will subscribe a term deposit from the Bank. <br>
================================================= <br>


New clients dataset:

<center>

| Column | Alex | Veronica | Pauline |
| --- | --- | --- | --- |
| `age` | 27 | 58 | 23 |
| `job` | blue-collar | retired | entrepreneur |
| `marital` | single | divorced | married |
| `education` | tertiary | secondary | unknown |
| `default` | no | yes | no |
| `balance` | 1500 | 9999 | -2000 |
| `housing` | no | no | yes |
| `loan` | yes | no | yes |
| `contact` | cellular | telephone | `--null-value--` |
| `day` | 30 | 15 | 5 |
| `month` | dec | nov | mar |
| `duration` | `--null-value--` | `--null-value--` | `--null-value--` |
| `campaign` | 1 | 3 | 0 |
| `pdays` | 180 | 35 | -1 |
| `previous` | 0 | 5 | 0 |
| `poutcome`| failure | success | unknown |

</center>

> The `duration` data all missing because the call is not yet performed, as explained in the main notebook

We will answer whether above clients are predicted to potentially subscribe to a term deposit products or not, with chosen machine learning model from `P1M2_fernando_namora.ipynb` with file of the pipeline model `term_depo_predictor.pkl`.

# ii. Answer

## Import Libraries

In [1]:
# import libraries used for this model inference project

import pickle
import pandas as pd
import numpy as np

# suppress warnings from pandas
import warnings
warnings.filterwarnings("ignore")

## Load Model

In [2]:
# load the model file

# alongwith the pipeline
with open('term_depo_predictor.pkl', 'rb') as file:
    best_svm = pickle.load(file)

best_svm

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipe_cat_ord',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinalencoder',
                                                                   OrdinalEncoder())]),
                                                  ['month']),
                                                 ('pipe_cat_nom',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['job', 'education',
                                                   'housing', 'loan',
                                                   'poutcome']),
                                                 ('pipe_num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('winsorizer',
                                                                   Winsorizer(capping_method='iqr',
                                                                              fold=1.5,
                                                                              tail='both')),
                                                                  ('minmaxscaler',
                                                                   MinMaxScaler())]),
                                                  ['age'])])),
                ('svc', SVC(C=0.1, gamma=1))])

## Inferencing

In [3]:
# define new data
# use all columns not just the results of feature selection

data_inf = pd.DataFrame({
    'age' : [27, 58, 23],
    'job' : ['blue-collar', 'retired', 'entrepreneur'],
    'marital' : ['single', 'divorced', 'married'],
    'education' : ['tertiary', 'secondary', 'unknown'],
    'default' : ['no', 'yes', 'no'],
    'balance' : [1500, 9999, -2000],
    'housing' : ['no', 'no', 'yes'],
    'loan' : ['yes', 'no', 'yes'],
    'contact' : ['cellular', 'telephone', np.nan],
    'day' : [30, 15, 5],
    'month' : ['dec', 'nov', 'mar'],
    'duration' : [np.nan, np.nan, np.nan], # since the calls not yet occurred
    'campaign' : [1, 3, 0],
    'pdays' : [180, 35, -1],
    'previous' : [0, 5, 0],
    'poutcome' : ['failure', 'success', 'unknown']
}, index = ['Alex', 'Veronica', 'Pauline'])

data_inf

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
Alex,27,blue-collar,single,tertiary,no,1500,no,yes,cellular,30,dec,NaN,1,180,0,failure
Veronica,58,retired,divorced,secondary,yes,9999,no,no,telephone,15,nov,NaN,3,35,5,success
Pauline,23,entrepreneur,married,unknown,no,-2000,yes,yes,NaN,5,mar,NaN,0,-1,0,unknown


Since column `job`, `day`, and `month` are previously re-grouped in the main notebook, we will do the same in here before further process :

In [4]:
# define same function on the main notebook

## function for re-grouping column `job`
def job_top5_grouping(df, col='job'):
    
    # list the top 5 from such column
    job_top5 = ['blue-collar', 'management', 'technician', 'admin.', 'services']

    # change other values into 'other'
    df.loc[~(df[col].isin(job_top5)), col] = 'other'   

## function for re-grouping column `day`
def day_binning(series):
    return pd.cut(
        series,
        bins=[0, 6, 12, 18, 24, 31],
        labels=['1 - 6', '7 - 12', '13 - 18', '19 - 24', '25 - 31']
    )

## function for re-grouping column `month`
def month_binning(series):

    # map the month to number first
    month_number = {
        'jan' : 1, 'feb' : 2, 'mar' : 3, 'apr' : 4,
        'may' : 5, 'jun' : 6, 'jul' : 7, 'aug' : 8,
        'sep' : 9, 'oct' : 10, 'nov' : 11, 'dec' : 12
    }

    # map and replace column `month` with number
    series = series.map(month_number)

    return pd.cut(
        series,
        bins=[0, 3, 6, 9, 12],
        labels=['jan - mar', 'apr - jun', 'jul - sep', 'oct - dec']
    )

# regroup column `job`, `day`, and `month`
job_top5_grouping(data_inf, 'job')
data_inf['day'] = day_binning(data_inf['day'])
data_inf['month'] = month_binning(data_inf['month'])

# see the results
data_inf

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
Alex,27,blue-collar,single,tertiary,no,1500,no,yes,cellular,25 - 31,oct - dec,NaN,1,180,0,failure
Veronica,58,other,divorced,secondary,yes,9999,no,no,telephone,13 - 18,oct - dec,NaN,3,35,5,success
Pauline,23,other,married,unknown,no,-2000,yes,yes,NaN,1 - 6,jan - mar,NaN,0,-1,0,unknown


In [5]:
# load to the pipeline then predict
y_pred_inf = best_svm.predict(data_inf)

# create dataframe
y_pred_inf_df = pd.DataFrame(y_pred_inf, index = ['Alex', 'Veronica', 'Pauline'])
y_pred_inf_df.columns = [
    'Will the client subscribe a term deposit?'
]

# show the results
y_pred_inf_df

,Will the client subscribe a term deposit?
Alex,0
Veronica,1
Pauline,0


Based on our machine learning model, it is unfortunate that **Mr. Alex and Ms. Pauline are predicted will not subscribe** a term deposit products from the Bank. On the other hand, the good news is **Ms. Veronica is predicted will subscribe** a term deposit product, so it is worth for the next or current campaign to further approach her.